# LSTM 기반 다음 수면 단계 예측

직전 10분의 수면 단계 시퀀스로 다음 30초의 수면 단계 `W / N1 / N2 / N3 / REM`을 예측한다.

> 이 프로젝트는 EEG에서 현재 수면 단계를 판정하는 의료 모델이 아니다. 전문가가 판정한 과거 수면 단계가 주어졌을 때 다음 단계를 예측하는 시퀀스 모델이다.

## 1. 환경 설정

노트북은 프로젝트 루트(`Sleep_pattern/`)에서 실행한다. 필요한 패키지가 없다면 다음 셀의 주석을 해제한다.

In [ ]:
# 필요한 패키지가 없다면 아래 줄의 주석을 해제해 설치합니다.
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'train.py').exists():
    raise RuntimeError('프로젝트 루트에서 노트북을 실행하세요.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from train import (
    STAGES, SequenceDataset, SleepStageLSTM, combine_subjects,
    draw_confusion_matrix, load_subject_data, metrics_from_predictions,
    predict, set_seed, split_subjects, train_model,
)

SEED = 2026
SEQUENCE_LENGTH = 20
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
NOTEBOOK_RESULTS = PROJECT_ROOT / 'results_notebook'
NOTEBOOK_RESULTS.mkdir(exist_ok=True)
set_seed(SEED)
print('PyTorch:', torch.__version__)

## 2. Sleep-EDF Hypnogram 다운로드

전체 PSG 신호는 내려받지 않고 사람별 첫 번째 밤의 작은 Hypnogram 주석 파일만 사용한다.

In [ ]:
edf_files = list(DATA_DIR.glob('*-Hypnogram.edf'))
if len(edf_files) < 10:
    subprocess.run([
        sys.executable, 'src/download_data.py',
        '--output-dir', str(DATA_DIR),
        '--max-subjects', '40',
    ], check=True)

edf_files = sorted(DATA_DIR.glob('*-Hypnogram.edf'))
print(f'Hypnogram 파일 수: {len(edf_files)}')
print([path.name for path in edf_files[:5]])

## 3. 전처리 및 Sequence 생성

- 30초 단위 수면 단계를 `W, N1, N2, N3, REM`으로 변환한다.
- 기존 stage 3과 4는 N3로 통합한다.
- Movement와 Unknown이 포함된 Window는 제외한다.
- 첫 수면 전과 마지막 수면 후의 각성 구간은 각각 30분만 유지한다.
- 직전 20개 epoch(10분)를 입력으로, 바로 다음 epoch를 정답으로 사용한다.
- 겹치는 Window의 데이터 누수를 막기 위해 사람 단위로 분리한다.

In [ ]:
subject_data = load_subject_data(DATA_DIR, SEQUENCE_LENGTH)
train_subjects, val_subjects, test_subjects = split_subjects(
    list(subject_data), SEED
)
train_data = combine_subjects(subject_data, train_subjects)
val_data = combine_subjects(subject_data, val_subjects)
test_data = combine_subjects(subject_data, test_subjects)

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'subjects': [len(train_subjects), len(val_subjects), len(test_subjects)],
    'sequences': [len(train_data.y), len(val_data.y), len(test_data.y)],
})
display(split_summary)
assert not (set(train_subjects) & set(test_subjects))

In [ ]:
target_counts = np.bincount(train_data.y, minlength=len(STAGES))
class_distribution = pd.DataFrame({
    'stage': STAGES,
    'count': target_counts,
    'ratio': target_counts / target_counts.sum(),
})
display(class_distribution)

## 4. LSTM 입력과 모델

입력 구조는 `(batch_size, sequence_length, input_size) = (B, 20, 5)`이다.

- `B`: 한 번에 처리하는 Sequence 수
- `20`: 직전 20개 × 30초 = 10분
- `5`: W, N1, N2, N3, REM one-hot Feature

각 입력 window는 `t-20`부터 `t-1`까지의 단계이고, 정답은 바로 다음 30초 구간인 `t`의 단계다.

모델은 1층 LSTM(`hidden_size=32`)의 마지막 출력을 Dropout과 Linear layer에 전달한다.

In [ ]:
BATCH_SIZE = 256
HIDDEN_SIZE = 32
DROPOUT = 0.2
FORGET_GATE_BIAS = 1.0
LEARNING_RATE = 1e-3
MAX_EPOCHS = 50
PATIENCE = 7
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print('입력 예시 shape:', train_data.x[:BATCH_SIZE].shape)
print('Device:', DEVICE)
print(SleepStageLSTM(
    HIDDEN_SIZE, 1, DROPOUT, forget_gate_bias=FORGET_GATE_BIAS
))

## 5. 학습 함수

Loss는 제곱근 역빈도 가중 Cross Entropy이고, Optimizer는 Adam, learning rate는 `0.001`, 최대 epoch는 `50`, early stopping patience는 `7`이다.\n\n`transition_weight=1`이면 표준 LSTM, `transition_weight=4`이면 실제 단계가 바뀌는 Sequence를 네 배 자주 학습하는 전환 가중 LSTM이다.\n\nLSTM의 forget gate bias는 1로 초기화해 초기 학습에서 기존 상태를 보존하도록 유도한다. 매 epoch마다 출력되는 `grad_norm`은 gradient clipping 전 평균 L2 norm으로, 긴 시퀀스 학습에서 그래디언트 소실·폭주 여부를 점검하는 지표다.

In [ ]:
def run_experiment(transition_weight: float):
    set_seed(SEED)
    generator = torch.Generator().manual_seed(SEED)

    if transition_weight > 1:
        sample_weights = np.ones(len(train_data.y), dtype=np.float64)
        is_transition = train_data.y != train_data.x[:, -1]
        sample_weights[is_transition] = transition_weight
        sampler = WeightedRandomSampler(
            torch.as_tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
            generator=generator,
        )
        train_loader = DataLoader(
            SequenceDataset(train_data), batch_size=BATCH_SIZE, sampler=sampler
        )
    else:
        train_loader = DataLoader(
            SequenceDataset(train_data), batch_size=BATCH_SIZE,
            shuffle=True, generator=generator
        )

    val_loader = DataLoader(SequenceDataset(val_data), batch_size=BATCH_SIZE)
    test_loader = DataLoader(SequenceDataset(test_data), batch_size=BATCH_SIZE)

    counts = np.bincount(train_data.y, minlength=len(STAGES))
    weights = np.sqrt(len(train_data.y) / (len(STAGES) * np.maximum(counts, 1)))
    loss_fn = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE)
    )

    model = SleepStageLSTM(
        HIDDEN_SIZE, 1, DROPOUT, forget_gate_bias=FORGET_GATE_BIAS
    ).to(DEVICE)
    model, history = train_model(
        model, train_loader, val_loader, loss_fn, DEVICE,
        LEARNING_RATE, MAX_EPOCHS, PATIENCE,
    )
    y_true, y_pred = predict(model, test_loader, DEVICE)
    all_metrics = metrics_from_predictions(y_true, y_pred)

    transition_mask = test_data.y != test_data.x[:, -1]
    transition_metrics = metrics_from_predictions(
        test_data.y[transition_mask], y_pred[transition_mask]
    )
    return model, history, y_pred, all_metrics, transition_metrics

## 6. 표준 LSTM 학습

In [ ]:
standard_model, standard_history, standard_pred, standard_all, standard_transition = (
    run_experiment(transition_weight=1.0)
)
print('Standard LSTM accuracy:', standard_all['accuracy'])
print('Standard LSTM macro F1:', standard_all['macro_f1'])

## 7. 전환 가중 LSTM 학습

In [ ]:
weighted_model, weighted_history, weighted_pred, weighted_all, weighted_transition = (
    run_experiment(transition_weight=4.0)
)
print('Weighted LSTM accuracy:', weighted_all['accuracy'])
print('Weighted transition accuracy:', weighted_transition['accuracy'])

## 8. 기준모델 및 결과 비교

기준모델은 입력의 마지막 수면 단계가 다음에도 그대로 유지된다고 예측한다. 전체 성능과 실제 단계가 바뀐 구간의 성능을 분리해서 비교한다. 다중 분류 문제이므로 Accuracy와 클래스 불균형을 반영하는 Macro F1을 함께 사용한다.

In [ ]:
baseline_pred = test_data.x[:, -1]
baseline_all = metrics_from_predictions(test_data.y, baseline_pred)
transition_mask = test_data.y != test_data.x[:, -1]
baseline_transition = metrics_from_predictions(
    test_data.y[transition_mask], baseline_pred[transition_mask]
)

comparison = pd.DataFrame([
    {
        'model': 'Last-stage baseline',
        'all_accuracy': baseline_all['accuracy'],
        'all_macro_f1': baseline_all['macro_f1'],
        'transition_accuracy': baseline_transition['accuracy'],
        'transition_macro_f1': baseline_transition['macro_f1'],
    },
    {
        'model': 'Standard LSTM',
        'all_accuracy': standard_all['accuracy'],
        'all_macro_f1': standard_all['macro_f1'],
        'transition_accuracy': standard_transition['accuracy'],
        'transition_macro_f1': standard_transition['macro_f1'],
    },
    {
        'model': 'Transition-weighted LSTM',
        'all_accuracy': weighted_all['accuracy'],
        'all_macro_f1': weighted_all['macro_f1'],
        'transition_accuracy': weighted_transition['accuracy'],
        'transition_macro_f1': weighted_transition['macro_f1'],
    },
])
display(comparison.style.format({
    'all_accuracy': '{:.4f}', 'all_macro_f1': '{:.4f}',
    'transition_accuracy': '{:.4f}', 'transition_macro_f1': '{:.4f}',
}))
comparison.to_csv(NOTEBOOK_RESULTS / 'model_comparison.csv', index=False)

In [ ]:
weighted_matrix = np.asarray(weighted_all['confusion_matrix'])
confusion_path = NOTEBOOK_RESULTS / 'weighted_confusion_matrix.png'
draw_confusion_matrix(weighted_matrix, confusion_path)
display(Image(filename=str(confusion_path)))

## 9. 재현 실행 결과 해석

seed 2026, forget gate bias 1.0 설정으로 재학습한 결과는 다음과 같다.

| 모델 | 전체 Accuracy | 전체 Macro F1 | 전환 Accuracy | 전환 Macro F1 |
|---|---:|---:|---:|---:|
| 마지막 단계 기준모델 | **0.9143** | **0.8524** | 0.0000 | 0.0000 |
| 표준 LSTM | 0.9087 | 0.8501 | 0.1452 | 0.1089 |
| 전환 가중 LSTM | 0.8771 | 0.8135 | **0.3116** | **0.2512** |

전체 시퀀스의 약 91.4%는 다음 단계가 마지막 입력 단계와 같다. 따라서 단순 기준모델이 전체 성능에서는 가장 높다. 전환 Sequence를 네 배 자주 학습하면 전체 정확도는 낮아지지만 단계가 실제로 바뀌는 구간의 정확도는 크게 증가한다.

전환 가중 LSTM에서 REM Recall은 가장 높고 N3 Recall은 가장 낮았다. N3는 N2로 오분류되는 경우가 많았는데, 두 단계가 인접하고 N2 표본이 훨씬 많기 때문으로 해석할 수 있다.

## 10. 결론과 한계

LSTM은 수면 단계의 지속성과 전환 순서를 학습할 수 있지만, 과거 단계만으로 정확한 전환 시점을 예측하는 데 한계가 있었다. 전체 Accuracy 하나만 보고 모델을 평가하면 강한 단계 지속성 때문에 성능을 과대평가하게 된다. 따라서 마지막 단계 기준모델, Macro F1, 전환 구간 성능을 함께 제시해야 한다.

후속 개선에서는 밤의 경과 시간, 현재 단계 지속 시간, EEG/EOG 주파수 Feature를 입력에 추가할 수 있다.